# Lab 2: Email Sender Agent with Status Handling

**AI Agent Lab 2 of 8**

---

## What this lab does

In this lab you will build an Email Sender Agent that simulates the full lifecycle of sending an email. The agent validates the email address, attempts to send it, handles failures by retrying up to three times, and logs every attempt in a list. At the end it prints a clear delivery report showing what happened to each email. This teaches you how real systems deal with unreliable operations.

## What you will learn

**Python concepts:**
- How to use the `random` module to simulate unpredictable outcomes
- How to use `random.seed()` to make random results repeatable
- How to validate a string with simple checks (looking for `@` and `.`)
- How to write a retry loop using `while` and a counter variable
- How to build and append to a list of dictionaries (a log)

**Agent concepts:**
- How agents handle failure and retry (resilience)
- How to keep an audit log of agent actions
- How to separate validation, sending, and reporting into distinct steps
- How randomness in simulations mirrors real-world unreliability

## Important things to know before starting

- **What is `random.seed()`?** Setting a seed makes the random module give the same sequence of numbers every time. This means every student will see exactly the same output when they run the notebook.
- **What is a retry loop?** Some operations (like sending an email) can fail temporarily. A retry loop tries the same operation again a set number of times before giving up.
- **What is an audit log?** An audit log is a running record of every action the agent took, including successes and failures. It helps you understand what happened.
- **This lab runs completely offline.** The `send_email()` function does not actually send anything.

## How the agent works

```
Input (to, subject, body)
        |
        v
  Think: validate the email address
        |
        v
  Act: attempt to send (up to 3 retries on timeout)
        |
        v
  Log: record every attempt in the delivery log
        |
        v
  Report: print a final delivery summary
```

---
## Step 0: Libraries and Setup

This lab uses **only built-in Python libraries** - nothing to install, no internet needed.

We use:
- `random` - to simulate unpredictable send outcomes (sent, invalid address, server timeout)
- `datetime` - to timestamp each send attempt in the log
- `sys` - to print the Python version

**Important:** We call `random.seed(42)` so that every student gets the same results. The number 42 is just a choice - any integer works as a seed.

In [ ]:
# No installation needed - this lab uses only Python built-in libraries
# Optional (not needed here): # !pip install nothing

import sys

print('Setup check passed!')
print(f'Python version: {sys.version}')
print('All libraries are built into Python - no internet required.')

---
## Importing Libraries

| Library | What it is used for in this lab |
|---|---|
| `random` | Picks a random outcome for each send attempt (sent / invalid address / server timeout) |
| `datetime` | Records the time of each send attempt in the log |

In [ ]:
import random    # used to simulate random send outcomes
import datetime # used to timestamp log entries

# Set the random seed so output is the same every time the notebook is run
# Change this number to see different random outcomes
random.seed(42)

print('Libraries imported and random seed set to 42.')
print('The random seed ensures every student sees identical output.')

---
## Step 1: Define the Possible Send Outcomes

**What this block does:** We define a list called `list_of_possible_outcomes`. Each time the agent tries to send an email, it randomly picks one of these outcomes. The probabilities are weighted: sent is most likely, timeout is possible, invalid address is rare.

**Why we need it:** In the real world, email servers sometimes time out or reject addresses. By weighting the outcomes, we make the simulation feel realistic.

**What to expect as output:** A printed list of all possible outcomes.

In [ ]:
# These are the possible results of one send attempt.
# 'sent' appears 4 times so it is picked most often (4 out of 7 chance)
# 'failed: server timeout' appears twice (2 out of 7 chance)
# 'failed: invalid address' appears once (1 out of 7 chance)
# Using a weighted list is a simple way to control probabilities.
list_of_possible_outcomes = [
    'sent',
    'sent',
    'sent',
    'sent',
    'failed: server timeout',
    'failed: server timeout',
    'failed: invalid address'
]

# Maximum number of times the agent will retry a failed send
# This variable is used inside the retry loop in Step 4
maximum_retry_attempts = 3

print('Possible outcomes defined:')
for outcome in set(list_of_possible_outcomes):
    count = list_of_possible_outcomes.count(outcome)
    print(f'  {outcome!r}: appears {count} time(s) out of {len(list_of_possible_outcomes)}')

---
## Step 2: Build the Email Address Validator

**What this block does:** We define a function called `validate_email_address()`. It checks whether the address looks like a valid email by looking for an `@` symbol and a `.` after it. This is a simple check - real email validators are more complex, but this is enough for our lab.

**Why we need it:** Sending to an invalid address wastes time and resources. The agent should catch obvious mistakes before even trying to send.

**What to expect as output:** This cell defines the function and runs a quick test on two example addresses.

In [ ]:
def validate_email_address(email_address_to_check):
    """
    Checks whether an email address looks valid.

    A valid address must have an '@' symbol, and a '.' must appear
    somewhere after the '@' symbol.

    Takes in:
      email_address_to_check - the email address string to validate

    Returns:
      True if the address looks valid, False otherwise.
    """

    # Check that '@' exists somewhere in the address
    if '@' not in email_address_to_check:
        return False

    # Split on '@' and check the domain part (after '@') contains a dot
    parts_of_address = email_address_to_check.split('@')
    domain_part = parts_of_address[-1]  # everything after the '@'

    if '.' not in domain_part:
        return False

    # The address passed both checks
    return True


# Quick test of the validator - students can see it working before the full demo
print('Validator test results:')
print(f'  ahmed@example.com  -> valid: {validate_email_address("ahmed@example.com")}')
print(f'  not-an-email       -> valid: {validate_email_address("not-an-email")}')
print(f'  missing@nodot      -> valid: {validate_email_address("missing@nodot")}')

---
## Step 3: Build the Simulated Send Function

**What this block does:** We define `send_email_simulation()`. This function pretends to send an email by randomly picking an outcome from `list_of_possible_outcomes`. It also records a timestamp for the attempt.

**Why we need it:** This is the 'action' step of the agent. In a real system, this would call an email service API. Here, we simulate it so we can practice retry logic without needing internet.

**What to expect as output:** The function is defined. A quick single-send test is printed.

In [ ]:
# SIMULATED SERVICE: In a real system this would call an email sending API.
# Here, we randomly return one of the pre-defined outcomes.

def send_email_simulation(to_address, subject_line, email_body):
    """
    Simulates sending an email and returns the result.

    Takes in:
      to_address   - the recipient's email address (string)
      subject_line - the email subject (string)
      email_body   - the text content of the email (string)

    Returns:
      A tuple: (outcome_string, timestamp_string)
      outcome_string is one of: 'sent', 'failed: server timeout',
      'failed: invalid address'
    """

    # Pick a random outcome from our weighted list
    send_outcome = random.choice(list_of_possible_outcomes)

    # Record when this attempt happened
    attempt_timestamp = datetime.datetime.now().strftime('%H:%M:%S')

    return send_outcome, attempt_timestamp


# Quick demonstration of a single send attempt
test_outcome, test_time = send_email_simulation('test@example.com', 'Hello', 'Test body')
print(f'Single send test -> outcome: {test_outcome!r}, time: {test_time}')

---
## Step 4: Build the Agent with Retry Logic

**What this block does:** We define `email_sender_agent()`, which is the main agent function. It validates the address first. If valid, it tries to send the email up to `maximum_retry_attempts` times. It only retries on timeout errors - it does NOT retry if the address is invalid. Every attempt is recorded in `list_of_delivery_logs`.

**Why we need it:** This teaches a core concept in agent design: handling failures gracefully. Real-world systems do not crash on the first error - they retry and track what happened.

**What to expect as output:** This cell defines the function. No emails are sent yet.

In [ ]:
# This list stores a record of every send attempt across all emails.
# Each entry is a dictionary. It is printed as the delivery report in Step 6.
list_of_delivery_logs = []

def email_sender_agent(to_address, subject_line, email_body):
    """
    The main Email Sender Agent with validation and retry logic.

    Takes in:
      to_address   - the recipient email address (string)
      subject_line - the subject of the email (string)
      email_body   - the body text of the email (string)

    Returns:
      The final outcome string ('sent' or the failure reason).
    """

    print(f'\nSending to: {to_address}')
    print(f'Subject: {subject_line}')

    # --- STEP A: Validate the email address before attempting to send ---
    address_is_valid = validate_email_address(to_address)

    if not address_is_valid:
        # The address failed validation - do not even try to send
        print('  Agent: address failed validation - will not attempt to send.')
        log_entry = {
            'to': to_address,
            'subject': subject_line,
            'attempt_number': 0,
            'outcome': 'failed: invalid address (caught by validator)',
            'timestamp': datetime.datetime.now().strftime('%H:%M:%S')
        }
        list_of_delivery_logs.append(log_entry)
        return 'failed: invalid address (caught by validator)'

    # --- STEP B: Try to send, retrying on timeout ---
    number_of_attempts_made = 0   # counter for how many attempts we have made
    final_send_outcome = None     # stores the last outcome; set when the loop ends

    while number_of_attempts_made < maximum_retry_attempts:
        number_of_attempts_made = number_of_attempts_made + 1
        print(f'  Attempt {number_of_attempts_made} of {maximum_retry_attempts}...')

        # Call the simulated send function
        send_result, attempt_time = send_email_simulation(to_address, subject_line, email_body)
        final_send_outcome = send_result

        # Record this attempt in the delivery log
        log_entry = {
            'to': to_address,
            'subject': subject_line,
            'attempt_number': number_of_attempts_made,
            'outcome': send_result,
            'timestamp': attempt_time
        }
        list_of_delivery_logs.append(log_entry)

        print(f'    Result: {send_result}')

        # If sent or invalid address, stop immediately - no point retrying
        if send_result == 'sent':
            print('  Agent: email sent successfully!')
            return send_result
        elif 'invalid address' in send_result:
            print('  Agent: address rejected by server - will not retry.')
            return send_result
        else:
            # It was a timeout - we will retry
            print('  Agent: timeout detected - will retry...')

    # All retry attempts are exhausted
    print(f'  Agent: all {maximum_retry_attempts} attempts failed.')
    return final_send_outcome


print('email_sender_agent() function defined successfully.')
print(f'The agent will retry up to {maximum_retry_attempts} times on timeout.')

---
## Step 5: Define Sample Emails and Run the Agent

**What this block does:** We define a list of five sample emails with different addresses (some valid, one deliberately invalid) and run the agent on each one. The random seed is reset before the run so output is always repeatable.

**Why we need it:** This is the main demo. You will see the agent validate addresses, retry on timeouts, give up after three failures, and report all results.

**What to expect as output:** Printed attempt logs for each of the five emails.

In [ ]:
# Reset the random seed before running the full demo so output is always identical
random.seed(42)

# Clear any earlier log entries so we start with a fresh report
list_of_delivery_logs.clear()

# Each dictionary represents one email the agent needs to send
list_of_sample_emails_to_send = [
    {'to': 'alice@university.edu', 'subject': 'Lecture Notes Upload', 'body': 'Hi Alice, the notes are ready.'},
    {'to': 'bob@company.org',      'subject': 'Meeting Rescheduled', 'body': 'Hi Bob, moving the meeting to Friday.'},
    {'to': 'not-a-valid-email',    'subject': 'Test Bad Address', 'body': 'This should fail validation.'},
    {'to': 'carol@school.net',     'subject': 'Assignment Deadline', 'body': 'Hi Carol, deadline is tomorrow.'},
    {'to': 'dave@research.io',     'subject': 'Report Ready', 'body': 'Hi Dave, the report is attached.'}
]

print('=' * 60)
print('STARTING EMAIL SENDER AGENT')
print(f'Emails to process: {len(list_of_sample_emails_to_send)}')
print('=' * 60)

# Run the agent on every email in the list
for email_item in list_of_sample_emails_to_send:
    email_sender_agent(
        to_address=email_item['to'],
        subject_line=email_item['subject'],
        email_body=email_item['body']
    )

print('\nAll emails processed.')

---
## Step 6: Print the Delivery Report

**What this block does:** We loop over `list_of_delivery_logs` and print every attempt in a formatted report table. At the bottom we print totals: how many were sent, how many failed, and how many were invalid.

**Why we need it:** The delivery report is the final output of the agent's work. It gives the user a clear picture of what happened without them having to scroll through all the attempt messages.

**What to expect as output:** A formatted table of all log entries, followed by a summary.

In [ ]:
# Count outcomes for the summary line at the bottom
total_sent_count = 0
total_failed_count = 0
total_invalid_count = 0

print('=' * 70)
print('DELIVERY REPORT')
print('=' * 70)
print(f'{"To":<30} {"Attempt":<10} {"Outcome":<25} {"Time"}')
print('-' * 70)

# Loop through every logged attempt and print it as a row
for log_entry in list_of_delivery_logs:
    print(f"{log_entry['to']:<30} "
          f"{str(log_entry['attempt_number']):<10} "
          f"{log_entry['outcome']:<25} "
          f"{log_entry['timestamp']}")

    # Count the outcome type for the summary
    if log_entry['outcome'] == 'sent':
        total_sent_count = total_sent_count + 1
    elif 'invalid' in log_entry['outcome']:
        total_invalid_count = total_invalid_count + 1
    else:
        total_failed_count = total_failed_count + 1

print('=' * 70)
print('SUMMARY')
print(f'  Successful sends : {total_sent_count}')
print(f'  Timeout failures : {total_failed_count}')
print(f'  Invalid addresses: {total_invalid_count}')
print(f'  Total log entries: {len(list_of_delivery_logs)}')
print('=' * 70)

---
## Summary

You built an **Email Sender Agent with Status Handling** that validates addresses, retries on failure, and keeps a full audit log.

### Key takeaways
- **random.seed() makes results reproducible:** Always set a seed when using randomness in a demo or test.
- **Separate validation from sending:** Catching bad addresses before sending saves unnecessary network calls.
- **Retry only what makes sense:** We retry timeouts but not invalid addresses - retrying an invalid address would never succeed.
- **Logging is essential:** The delivery log lets you audit exactly what happened and when.
- **Lists of dictionaries are a simple database:** Each log entry is a dictionary; the whole log is a list. This pattern appears in many real systems.

### Connection to the next lab

In **Lab 3**, you will build an Email Classifier Agent. Instead of sending emails, you will give the agent a inbox of emails and it will sort them into categories (Urgent, Spam, Normal) using keyword rules.

---

## Try It Yourself

1. **Change the seed:** Change `random.seed(42)` to `random.seed(99)` and re-run the notebook. Notice how the outcomes change. Then change it back to 42 and confirm the original output returns.

2. **Increase max retries:** Change `maximum_retry_attempts` from 3 to 5. Re-run and observe whether more timeouts eventually succeed after extra retries.

3. **Add a new failure type:** Add `'failed: mailbox full'` to `list_of_possible_outcomes`. Update the retry logic so the agent does NOT retry on mailbox-full (just like invalid address). Think about why this would be the right behaviour.